In [1]:
import pickle
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import copy
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

import matplotlib.pyplot as plt

In [3]:
with open("simulated_snp_dataset.pkl", "rb") as f:
    dataset = pickle.load(f)
    
num_snps_list = [item["X"].shape[1] for item in dataset]
print(np.percentile(num_snps_list, [50, 75, 90, 95, 99]))

[ 1.  2.  4.  6. 13.]


In [6]:
def pad_matrix(G, max_n_samples=50, max_snps=200):
    n, s = G.shape
    out = np.zeros((max_n_samples, max_snps), dtype=np.float32)

    n_use = min(n, max_n_samples)
    s_use = min(s, max_snps)

    out[:n_use, :s_use] = G[:n_use, :s_use]
    return out

In [9]:
def extract_features(item):
    """
    Convert one simulated SNP dataset item into a fixed-length feature vector.
    
    item should contain:
      - item["X"]: SNP matrix of shape (n_samples, num_snps)
      - item["theta"]
      - item["metadata"]
    """
    G = item["X"]   # shape: (n_samples, num_snps)
    n, s = G.shape
    
    theta = item.get("theta", np.nan)
    seq_len = item.get("metadata", {}).get("sequence_length", np.nan)
    
    # Handle edge case: zero SNPs
    if s == 0:
        return {
            "n_samples": n,
            "num_snps": 0,
            "snp_density": 0.0,
            "theta": theta,
            "sequence_length": seq_len,
            "mean_af": 0.0,
            "std_af": 0.0,
            "min_af": 0.0,
            "max_af": 0.0,
            "mean_heterozygosity": 0.0,
            "std_heterozygosity": 0.0,
            "singleton_prop": 0.0,
            "doubleton_prop": 0.0,
            "mean_minor_ac": 0.0,
            "var_minor_ac": 0.0,
            "mean_pairwise_diff": 0.0,
        }
    
    # Derived allele frequency per SNP
    allele_counts = G.sum(axis=0)                 # shape: (num_snps,)
    af = allele_counts / n                        # allele frequency
    minor_ac = np.minimum(allele_counts, n - allele_counts)
    
    # Per-site heterozygosity proxy
    heterozygosity = 2 * af * (1 - af)
    
    singleton_prop = np.mean((allele_counts == 1) | (allele_counts == n - 1))
    doubleton_prop = np.mean((allele_counts == 2) | (allele_counts == n - 2))
    
    # Mean pairwise difference proxy:
    # For a biallelic site, expected pairwise difference = 2p(1-p)
    mean_pairwise_diff = np.sum(heterozygosity)
    
    return {
        "n_samples": n,
        "num_snps": s,
        "snp_density": s / seq_len if seq_len and seq_len > 0 else np.nan,
        "theta": theta,
        "sequence_length": seq_len,
        "mean_af": np.mean(af),
        "std_af": np.std(af),
        "min_af": np.min(af),
        "max_af": np.max(af),
        "mean_heterozygosity": np.mean(heterozygosity),
        "std_heterozygosity": np.std(heterozygosity),
        "singleton_prop": singleton_prop,
        "doubleton_prop": doubleton_prop,
        "mean_minor_ac": np.mean(minor_ac),
        "var_minor_ac": np.var(minor_ac),
        "mean_pairwise_diff": mean_pairwise_diff,
    }

In [10]:
def evaluate_model(model, loader, device):
    model.eval()
    ys, preds = [], []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            pred = model(X_batch)

            ys.append(y_batch.cpu().numpy())
            preds.append(pred.cpu().numpy())

    y_true = np.vstack(ys).ravel()
    y_pred = np.vstack(preds).ravel()

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    corr = np.corrcoef(y_true, y_pred)[0, 1]

    return {
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "corr": corr,
        "y_true": y_true,
        "y_pred": y_pred
    }


def train_mlp(model, train_loader, val_loader, device, lr=1e-3, epochs=50):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_rmse = float("inf")
    best_state = None

    train_losses = []
    val_rmses = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            pred = model(X_batch)
            loss = criterion(pred, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(X_batch)

        train_loss = running_loss / len(train_loader.dataset)
        val_metrics = evaluate_model(model, val_loader, device)

        train_losses.append(train_loss)
        val_rmses.append(val_metrics["rmse"])

        print(
            f"Epoch {epoch+1:03d} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val RMSE: {val_metrics['rmse']:.6f} | "
            f"Val Corr: {val_metrics['corr']:.4f}"
        )

        if val_metrics["rmse"] < best_val_rmse:
            best_val_rmse = val_metrics["rmse"]
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return train_losses, val_rmses

In [11]:
# matrix branch
X_mats = np.stack([
    pad_matrix(item["X"], max_n_samples=50, max_snps=200)
    for item in dataset
])[:, None, :, :]

# summary branch
rows = [extract_features(item) for item in dataset]
X_tab = pd.DataFrame(rows).fillna(0).values.astype(np.float32)

# scale tabular features
from sklearn.preprocessing import StandardScaler
tab_scaler = StandardScaler()
X_tab = tab_scaler.fit_transform(X_tab).astype(np.float32)

y = np.array([item["alpha"] for item in dataset], dtype=np.float32)

In [12]:
idx = np.arange(len(y))
idx_trainval, idx_test = train_test_split(idx, test_size=0.15, random_state=42)
idx_train, idx_val = train_test_split(idx_trainval, test_size=0.15, random_state=42)

Xmat_train, Xmat_val, Xmat_test = X_mats[idx_train], X_mats[idx_val], X_mats[idx_test]
Xtab_train, Xtab_val, Xtab_test = X_tab[idx_train], X_tab[idx_val], X_tab[idx_test]
y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]

In [13]:
class HybridDataset(Dataset):
    def __init__(self, Xmat, Xtab, y):
        self.Xmat = torch.tensor(Xmat, dtype=torch.float32)
        self.Xtab = torch.tensor(Xtab, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.Xmat[idx], self.Xtab[idx], self.y[idx]

In [14]:
class HybridSNPModel(nn.Module):
    def __init__(self, tab_dim):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.cnn_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 6 * 25, 128),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.tab_head = nn.Sequential(
            nn.Linear(tab_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU()
        )

        self.final_head = nn.Sequential(
            nn.Linear(128 + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 1)
        )

    def forward(self, xmat, xtab):
        a = self.cnn(xmat)
        a = self.cnn_head(a)

        b = self.tab_head(xtab)

        x = torch.cat([a, b], dim=1)
        return self.final_head(x)

In [15]:
def evaluate_hybrid(model, loader, device):
    model.eval()
    ys, preds = [], []

    with torch.no_grad():
        for xmat, xtab, y_batch in loader:
            xmat = xmat.to(device)
            xtab = xtab.to(device)
            y_batch = y_batch.to(device)

            pred = model(xmat, xtab)

            ys.append(y_batch.cpu().numpy())
            preds.append(pred.cpu().numpy())

    y_true = np.vstack(ys).ravel()
    y_pred = np.vstack(preds).ravel()

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    corr = np.corrcoef(y_true, y_pred)[0, 1]

    return {
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "corr": corr,
        "y_true": y_true,
        "y_pred": y_pred
    }


def train_hybrid(model, train_loader, val_loader, device, lr=1e-3, epochs=30):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_rmse = float("inf")
    best_state = None
    train_losses = []
    val_rmses = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for xmat, xtab, y_batch in train_loader:
            xmat = xmat.to(device)
            xtab = xtab.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            pred = model(xmat, xtab)
            loss = criterion(pred, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(xmat)

        train_loss = running_loss / len(train_loader.dataset)
        val_metrics = evaluate_hybrid(model, val_loader, device)

        train_losses.append(train_loss)
        val_rmses.append(val_metrics["rmse"])

        print(
            f"Epoch {epoch+1:03d} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val RMSE: {val_metrics['rmse']:.6f} | "
            f"Val Corr: {val_metrics['corr']:.4f}"
        )

        if val_metrics["rmse"] < best_val_rmse:
            best_val_rmse = val_metrics["rmse"]
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return train_losses, val_rmses

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_ds = HybridDataset(Xmat_train, Xtab_train, y_train)
val_ds = HybridDataset(Xmat_val, Xtab_val, y_val)
test_ds = HybridDataset(Xmat_test, Xtab_test, y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

hybrid_model = HybridSNPModel(tab_dim=Xtab_train.shape[1]).to(device)
train_losses, val_rmses = train_hybrid(
    hybrid_model, train_loader, val_loader, device, lr=1e-3, epochs=30
)

hybrid_test_metrics = evaluate_hybrid(hybrid_model, test_loader, device)
print("\nHybrid Test Performance")
print(hybrid_test_metrics["rmse"], hybrid_test_metrics["mae"], hybrid_test_metrics["r2"], hybrid_test_metrics["corr"])

Epoch 001 | Train Loss: 0.000312 | Val RMSE: 0.012259 | Val Corr: 0.5614
Epoch 002 | Train Loss: 0.000157 | Val RMSE: 0.012029 | Val Corr: 0.5666
Epoch 003 | Train Loss: 0.000150 | Val RMSE: 0.011983 | Val Corr: 0.5736
Epoch 004 | Train Loss: 0.000147 | Val RMSE: 0.011951 | Val Corr: 0.5749
Epoch 005 | Train Loss: 0.000147 | Val RMSE: 0.011955 | Val Corr: 0.5722
Epoch 006 | Train Loss: 0.000146 | Val RMSE: 0.012008 | Val Corr: 0.5755
Epoch 007 | Train Loss: 0.000145 | Val RMSE: 0.012125 | Val Corr: 0.5768
Epoch 008 | Train Loss: 0.000145 | Val RMSE: 0.011895 | Val Corr: 0.5758
Epoch 009 | Train Loss: 0.000145 | Val RMSE: 0.012135 | Val Corr: 0.5746
Epoch 010 | Train Loss: 0.000145 | Val RMSE: 0.011916 | Val Corr: 0.5762
Epoch 011 | Train Loss: 0.000145 | Val RMSE: 0.011945 | Val Corr: 0.5734
Epoch 012 | Train Loss: 0.000144 | Val RMSE: 0.011907 | Val Corr: 0.5741
Epoch 013 | Train Loss: 0.000143 | Val RMSE: 0.012031 | Val Corr: 0.5764
Epoch 014 | Train Loss: 0.000143 | Val RMSE: 0.0119